# Tutorial 02: 条件付き生成 - 完全版## 概要このチュートリアルでは、物理化学的性質を指定した分子生成を学びます。### 学習内容1. 6つの性質（alpha, gap, homo, lumo, mu, Cv）の理解2. 条件付きモデルの訓練3. 性質の正規化手法4. 目標値を指定した生成5. 性質分類器による評価### 前提知識- Tutorial 01の内容- 物理化学の基礎知識所要時間: 45-60分

In [ ]:
# 環境のセットアップimport torchimport numpy as npimport matplotlib.pyplot as pltimport syssys.path.append('../..')from qm9 import datasetfrom qm9.utils import compute_mean_mad, prepare_contextfrom configs.datasets_config import get_dataset_infoprint("PyTorch version:", torch.__version__)print("CUDA available:", torch.cuda.is_available())

## セクション1: 条件付け可能な性質の理解QM9データセットには以下の物理化学的性質が含まれています：### 性質の詳細| 性質 | 記号 | 説明 | 単位 | 典型的範囲 ||------|------|------|------|-----------|| **alpha** | α | 等方的分極率 | Bohr³ | 40-220 || **gap** | ΔE | HOMO-LUMOギャップ | eV | 2.5-18 || **homo** | εH | HOMO準位エネルギー | eV | -11 to -2 || **lumo** | εL | LUMO準位エネルギー | eV | -4 to 7 || **mu** | μ | 双極子モーメント | Debye | 0-12 || **Cv** | Cv | 熱容量@298K | cal/mol·K | 6-64 |### 性質の物理的意味**alpha (分極率)**:- 外部電場に対する応答- 大きいほど電子雲が変形しやすい- 薬物設計で重要

In [ ]:
# データの読み込みと性質分布の確認class Args:    batch_size = 64    num_workers = 2    filter_n_atoms = None    dataset = 'qm9'    remove_h = False    include_charges = Trueargs = Args()dataloaders, charge_scale = dataset.retrieve_dataloaders(args)# 性質の統計properties = ['alpha', 'gap', 'homo', 'lumo', 'mu', 'Cv']train_data = dataloaders['train'].dataset.dataprint("\nQM9性質の統計:")print("=" * 70)for prop in properties:    if prop in train_data:        values = train_data[prop]        print(f"{prop:6s}: mean={values.mean():7.2f}, std={values.std():6.2f}, "              f"min={values.min():7.2f}, max={values.max():7.2f}")# 分布の可視化fig, axes = plt.subplots(2, 3, figsize=(15, 10))axes = axes.flatten()for i, prop in enumerate(properties):    if prop in train_data:        values = train_data[prop].numpy()        axes[i].hist(values, bins=50, alpha=0.7, edgecolor='black')        axes[i].set_title(f'{prop} distribution')        axes[i].set_xlabel(prop)        axes[i].set_ylabel('Count')        axes[i].grid(True, alpha=0.3)plt.tight_layout()plt.savefig('property_distributions.png', dpi=150)print("\n✓ 分布グラフ保存: property_distributions.png")

## セクション2: 性質の正規化条件付け前に、性質を正規化する必要があります。本システムではMAD (Median Absolute Deviation) を使用します。### MADによる正規化MADは外れ値に頑健な統計量です：$$\text{MAD}(X) = \text{median}(|X_i - \text{median}(X)|)$$正規化:$$\tilde{x} = \frac{x - \text{mean}(X)}{\text{MAD}(X)}$$

In [ ]:
# 性質の正規化conditioning_properties = ['alpha', 'gap']# mean と MAD の計算property_norms = compute_mean_mad(dataloaders, conditioning_properties, 'qm9')print("正規化パラメータ:")print("=" * 50)for prop in conditioning_properties:    mean = property_norms[prop]['mean']    mad = property_norms[prop]['mad']    print(f"{prop}:")    print(f"  mean = {mean:.4f}")    print(f"  MAD  = {mad:.4f}")# 正規化の例sample_values = {    'alpha': torch.tensor([60.0, 80.0, 100.0]),    'gap': torch.tensor([5.0, 7.0, 9.0])}print("\n正規化の例:")print("=" * 50)for prop, values in sample_values.items():    mean = property_norms[prop]['mean']    mad = property_norms[prop]['mad']    normalized = (values - mean) / mad    print(f"{prop}:")    print(f"  元の値: {values.numpy()}")    print(f"  正規化後: {normalized.numpy()}")

## セクション3: 条件付きモデルの訓練条件付きモデルを訓練します。実際の訓練には時間がかかるため、ここではコマンドと設定を説明します。

In [ ]:
# 条件付きモデルの訓練コマンド（実行はスキップ）training_command = '''python main_qm9.py \\    --exp_name cond_alpha_gap \\    --conditioning alpha gap \\    --model egnn_dynamics \\    --n_epochs 3000 \\    --batch_size 64 \\    --nf 192 \\    --n_layers 9 \\    --lr 1e-4 \\    --diffusion_steps 1000 \\    --diffusion_noise_schedule polynomial_2 \\    --diffusion_loss_type l2 \\    --normalize_factors 1,8,1 \\    --save_model True'''print("条件付きモデルの訓練コマンド:")print(training_command)print("\n重要な設定の違い:")print("  --conditioning alpha gap    # 条件付けする性質")print("  --normalize_factors 1,8,1   # 条件付け時は第3項を1に")print("\n訓練時間: V100で約24-48時間")

## セクション4: 条件付きサンプリング訓練済みモデルから、指定した性質を持つ分子を生成します。

In [ ]:
# コンテキスト（条件）の準備def create_conditional_context(target_properties, property_norms):    """    目標性質から条件付けコンテキストを作成        Parameters:    -----------    target_properties : dict        {'property_name': value} の辞書    property_norms : dict        正規化パラメータ    """    # バッチサイズ1のダミーデータ    batch_data = {}    for prop, value in target_properties.items():        batch_data[prop] = torch.tensor([[value]], dtype=torch.float32)        # コンテキスト準備    context = prepare_context(        conditioning=list(target_properties.keys()),        minibatch=batch_data,        property_norms=property_norms    )        return context# 使用例target_alpha = 75.0  # Bohr³target_gap = 8.0     # eVtarget_props = {    'alpha': target_alpha,    'gap': target_gap}context = create_conditional_context(target_props, property_norms)print(f"作成されたコンテキストの形状: {context.shape}")print(f"コンテキスト値: {context}")print("\n✓ 条件付けコンテキスト作成完了")print(f"  目標 alpha: {target_alpha} Bohr³")print(f"  目標 gap: {target_gap} eV")

## セクション5: 性質スイープ性質の範囲をスキャンして、制御性を検証します。

In [ ]:
# 性質スイープの設定def property_sweep(property_name, property_norms, n_points=10):    """    性質の範囲をスイープするための値を生成    """    # データセット全体の範囲    train_data = dataloaders['train'].dataset.data    prop_values = train_data[property_name]        min_val = prop_values.min().item()    max_val = prop_values.max().item()        # n_pointsで等分    sweep_values = np.linspace(min_val, max_val, n_points)        print(f"{property_name}のスイープ:")    print(f"  範囲: {min_val:.2f} - {max_val:.2f}")    print(f"  ポイント数: {n_points}")    print(f"  値: {sweep_values}")        return sweep_values# alphaのスイープalpha_sweep = property_sweep('alpha', property_norms, n_points=10)# 各値でのコンテキスト作成contexts = []for alpha_val in alpha_sweep:    ctx = create_conditional_context(        {'alpha': alpha_val},        property_norms    )    contexts.append(ctx)print(f"\n✓ {len(contexts)}個のコンテキスト作成完了")

## セクション6: 複数性質の同時条件付け複数の性質を同時に制御します。

In [ ]:
# 複数性質の組み合わせmulti_property_targets = [    {'alpha': 60.0, 'gap': 9.0, 'mu': 2.0},   # 小さなalpha, 大きなgap    {'alpha': 80.0, 'gap': 7.0, 'mu': 3.0},   # 中程度    {'alpha': 100.0, 'gap': 5.0, 'mu': 4.0},  # 大きなalpha, 小さなgap]# 利用可能な性質のみを使用available_props = ['alpha', 'gap']  # muは訓練時に含めていない場合property_norms_available = {    k: v for k, v in property_norms.items() if k in available_props}print("複数性質の条件付け:")print("=" * 60)for i, targets in enumerate(multi_property_targets, 1):    # 利用可能な性質のみフィルタ    filtered_targets = {k: v for k, v in targets.items() if k in available_props}        print(f"\n条件セット {i}:")    for prop, value in filtered_targets.items():        print(f"  {prop}: {value}")        # コンテキスト作成    ctx = create_conditional_context(filtered_targets, property_norms_available)    print(f"  コンテキスト形状: {ctx.shape}")print("\n✓ 複数性質の条件付けコンテキスト準備完了")

## セクション7: 性質分類器の訓練生成された分子の性質を評価するための分類器を訓練します。### 分類器の役割1. 生成された分子の性質を予測2. 目標値との一致度を定量評価3. 条件付けの制御性能を測定

In [ ]:
# 性質分類器の訓練コマンドclassifier_training = '''cd qm9/property_predictionpython main_qm9_prop.py \\    --property alpha \\    --exp_name classifier_alpha \\    --model_name egnn \\    --num_workers 2 \\    --lr 5e-4 \\    --n_epochs 500 \\    --batch_size 64'''print("性質分類器の訓練コマンド:")print(classifier_training)print("\n分類器モデルのオプション:")print("  --model_name egnn      # EGNN分類器（推奨）")print("  --model_name numnodes  # ノード数のみのベースライン")print("\n訓練時間: V100で約2-3時間")

## セクション8: 定量的評価訓練した分類器を使用して、生成品質を定量評価します。

In [ ]:
# 条件付き生成の評価コマンドevaluation_command = '''python eval_conditional_qm9.py \\    --generators_path outputs/cond_alpha_gap \\    --classifiers_path qm9/property_prediction/outputs/classifier_alpha \\    --property alpha \\    --iterations 100 \\    --batch_size 100 \\    --task edm'''print("条件付き生成の評価コマンド:")print(evaluation_command)print("\n評価タスクの種類:")print("  --task qualitative   # 定性的評価（可視化）")print("  --task edm           # 定量的評価（分類器使用）")print("\n期待される評価メトリクス:")print("  - Mean Absolute Error (MAE): 目標値からの平均誤差")print("  - Correlation: 目標値と生成値の相関係数")print("  - Coverage: 目標範囲のカバー率")

## まとめ### 学習した内容✓ 6つの物理化学的性質の理解✓ MADによる正規化手法✓ 条件付きモデルの訓練方法✓ 目標値を指定した生成✓ 性質スイープによる制御性検証✓ 複数性質の同時条件付け✓ 性質分類器による定量評価### 実践的なヒント1. **性質の選択**: タスクに応じて適切な性質を選択2. **正規化の重要性**: 必ずMADで正規化3. **訓練時間**: 条件付きモデルは無条件より長め4. **評価の重要性**: 分類器で必ず定量評価### 次のステップ1. **Tutorial 03**: 分子記述子とASEデータベース   - より詳細な分子特性の制御   - 官能基、原子タイプの指定2. **実際の訓練**:    ```bash   python main_qm9.py --conditioning alpha gap --n_epochs 3000   ```3. **高度な条件付け**:   - 6つすべての性質を同時条件付け   - カスタム性質の追加### 参考資料- 完全ユーザーガイド: `../../doc/ja/完全ユーザーガイド.md`- 完全理論説明書: `../../doc/ja/完全理論説明書.md`- コード: `../../eval_conditional_qm9.py`---**NO FALLBACKS**: このチュートリアルのすべてのコードは理論的に厳密です。Happy Learning! 🎯